# Week 4: Eigenvalues, Eigenvectors & Applications
*TBIL Sections: GT1–GT4, Appendix A  |  Topics: Determinants, eigenvalues, PageRank*

---
> **How to use:**  
> • Click **`Copy to Drive`** at the top to save your own editable copy  
> • Run each cell with **`Shift + Enter`** or the ▶ button  
> • Complete the **Exercises** section at the bottom  


## Setup — Run This First


Run this cell before anything else.


In [ ]:
import numpy as np
from sympy import Matrix, symbols, factor, solve, eye
import matplotlib.pyplot as plt
print('Libraries loaded!')


## Part 1: Determinants (GT1, GT2)


The **determinant** measures how much a transformation **scales area/volume**:  
- |det| = 2 → areas double  
- det = 0 → the transformation collapses space (not invertible!)  
- det < 0 → orientation is flipped  


In [ ]:
# NumPy (fast, numerical)
A_np = np.array([[2.,1.,3.],[0.,-1.,4.],[5.,2.,0.]])
print(f'det(A) [NumPy] = {np.linalg.det(A_np):.4f}')

# SymPy (exact)
A_sym = Matrix([[2,1,3],[0,-1,4],[5,2,0]])
print(f'det(A) [SymPy] = {A_sym.det()}')

# Geometric meaning: area of parallelogram
v1 = np.array([3., 1.])
v2 = np.array([1., 2.])
M  = np.column_stack([v1, v2])
area = abs(np.linalg.det(M))
print(f'\nParallelogram area = |det| = {area:.2f}')

# Visualize
fig, ax = plt.subplots(figsize=(5,5))
para = np.array([[0,0],v1,v1+v2,v2,[0,0]])
ax.fill(para[:,0],para[:,1],alpha=0.3,color='blue',label=f'Area = {area:.1f}')
ax.quiver(0,0,v1[0],v1[1],angles='xy',scale_units='xy',scale=1,color='red',width=0.015,label=f'v1={v1}')
ax.quiver(0,0,v2[0],v2[1],angles='xy',scale_units='xy',scale=1,color='green',width=0.015,label=f'v2={v2}')
ax.set_xlim(-0.5,5); ax.set_ylim(-0.5,4)
ax.axhline(0,color='k',lw=0.5); ax.axvline(0,color='k',lw=0.5)
ax.grid(True,alpha=0.3); ax.legend(); ax.set_aspect('equal')
ax.set_title('Determinant = Area of Parallelogram')
plt.show()


## Part 2: Eigenvalues and Eigenvectors (GT3, GT4)


An **eigenvector** is a special direction that the transformation only **scales** (doesn't rotate).  
The scale factor is the **eigenvalue** λ:  
> **A·v = λ·v**  

To find eigenvalues, solve the **characteristic equation**:  
> **det(A − λI) = 0**  


In [ ]:
A = Matrix([[3, 1],[0, 2]])
lam = symbols('lambda')

char_poly = (A - lam*eye(2)).det()
print(f'Characteristic polynomial: {char_poly}')
print(f'Factored: {factor(char_poly)}')

eigenvalues = solve(char_poly, lam)
print(f'Eigenvalues: {eigenvalues}')

for lam_val in eigenvalues:
    print(f'\n--- Eigenspace for λ = {lam_val} ---')
    null_basis = (A - lam_val*eye(2)).nullspace()
    print(f'Eigenvector(s): {[list(v.T) for v in null_basis]}')


Now let's **verify** with NumPy and **visualize** the eigenvectors:


In [ ]:
A_np = np.array([[3.,1.],[0.,2.]])
eigenvalues_np, eigenvectors_np = np.linalg.eig(A_np)
print('Eigenvalues:', eigenvalues_np)
print('Eigenvectors (columns):\n', eigenvectors_np)

# Verify A @ v = lambda * v
for i in range(2):
    lv = eigenvalues_np[i]
    v  = eigenvectors_np[:, i]
    print(f'λ={lv:.1f}: A@v={np.round(A_np@v,4)}, λ*v={np.round(lv*v,4)}, match={np.allclose(A_np@v,lv*v)}')

# Visualize
fig, ax = plt.subplots(figsize=(6,6))
colors = ['red','blue']
for i in range(2):
    v = eigenvectors_np[:,i]
    Av = A_np @ v
    lv = eigenvalues_np[i]
    ax.quiver(0,0,v[0],v[1],angles='xy',scale_units='xy',scale=1,color=colors[i],width=0.02,label=f'eigenvec (λ={lv:.1f})')
    ax.quiver(0,0,Av[0],Av[1],angles='xy',scale_units='xy',scale=1,color=colors[i],width=0.015,alpha=0.4,label=f'A*v (scaled by λ)')
ax.set_xlim(-2,3.5); ax.set_ylim(-2,3.5)
ax.axhline(0,color='k',lw=0.5); ax.axvline(0,color='k',lw=0.5)
ax.grid(True,alpha=0.3); ax.legend(fontsize=9); ax.set_aspect('equal')
ax.set_title('Eigenvectors: A·v = λ·v  (only scaled, never rotated)')
plt.show()


## Part 3: Application — PageRank (Appendix A.2)


**PageRank** (Google's original algorithm) assigns importance scores to web pages.  
The scores are the **dominant eigenvector** (eigenvalue = 1) of a link matrix.

**4-page web:**  Page 1→{2,3,4},  Page 2→{3},  Page 3→{1,2},  Page 4→{1}


In [ ]:
# Column-stochastic link matrix
# Column j = how page j distributes its 'votes' equally among its outbound links
L = np.array([
    [0,    0,   1/2, 1  ],  # page 1
    [1/3,  0,   1/2, 0  ],  # page 2
    [1/3,  1,   0,   0  ],  # page 3
    [1/3,  0,   0,   0  ]   # page 4
])

eigenvalues, eigenvectors = np.linalg.eig(L)
idx = np.argmin(np.abs(eigenvalues - 1.0))  # find eigenvalue closest to 1
pagerank = eigenvectors[:, idx].real
pagerank = pagerank / pagerank.sum()         # normalize

print('PageRank scores:')
for i, pr in enumerate(pagerank):
    print(f'  Page {i+1}: {pr:.4f}')

# Bar chart
fig, ax = plt.subplots(figsize=(6,4))
bars = ax.bar([f'Page {i+1}' for i in range(4)], pagerank,
              color=['#4285F4','#EA4335','#FBBC05','#34A853'])
for bar, score in zip(bars, pagerank):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{score:.3f}', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('PageRank Score')
ax.set_title('PageRank: Dominant Eigenvector of Link Matrix')
plt.tight_layout(); plt.show()


## Part 4: Exercises — Your Turn! ✏️


**Exercise 1:** For `A = [[4,1],[2,3]]`:  
(a) Compute det(A) using SymPy.  (b) Find eigenvalues.  (c) Find eigenvectors.


In [ ]:
# Exercise 1
A_ex1 = Matrix([[4,1],[2,3]])
lam = symbols('lambda')
# det: A_ex1.det()
# char poly: (A_ex1 - lam*eye(2)).det()
# eigenvalues: solve(...)
# eigenvectors: (A_ex1 - lam_val*eye(2)).nullspace()


**Exercise 2:** Let `A = [[1,2],[2,4]]`.  
(a) What is det(A)?  (b) What does this tell you about eigenvalues?  (c) Find the eigenvector for λ=0.


In [ ]:
# Exercise 2
A_ex2 = Matrix([[1,2],[2,4]])
print('det =', A_ex2.det())
# Find eigenvector for lambda=0:
# (A_ex2 - 0*eye(2)).nullspace()


**Exercise 3 (PageRank extension):** Add a 5th page with links: Page 5→{2,4}, and Page 1 also links to Page 5.  
Update the link matrix and re-run PageRank. How does the ranking change?


In [ ]:
# Exercise 3 -- build a 5x5 link matrix
# Page 1 -> Pages 2,3,4,5 (1/4 each)
# Page 2 -> Page 3       (1)
# Page 3 -> Pages 1,2    (1/2 each)
# Page 4 -> Page 1       (1)
# Page 5 -> Pages 2,4    (1/2 each)

L5 = np.array([
    [0,    0,   1/2, 1,   0  ],
    [1/4,  0,   1/2, 0,   1/2],
    [1/4,  1,   0,   0,   0  ],
    [1/4,  0,   0,   0,   1/2],
    [1/4,  0,   0,   0,   0  ]
])

eigenvalues5, eigenvectors5 = np.linalg.eig(L5)
idx5 = np.argmin(np.abs(eigenvalues5 - 1.0))
pr5 = eigenvectors5[:, idx5].real
pr5 = pr5 / pr5.sum()
print('New PageRank scores (5 pages):')
for i, pr in enumerate(pr5):
    print(f'  Page {i+1}: {pr:.4f}')
